In [1]:
!pip install transformers -U
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/12.0 MB 9.4 MB/s eta 0:00:02
   ----------- ---------------------------- 3.4/12.0 MB 8.8 MB/s eta 0:00:01
   ------------------ --------------------- 5.5/12.0 MB 9.1 MB/s eta 0:00:01
   ----------------------- ---------------- 7.1/12.0 MB 8.6 MB/s eta 0:00:01
   ------------------------- -------------- 7.6/12.0 MB 7.3 MB/s eta 0:00:01
   --------------------------- ------------ 8.1/12.0 MB 6.6 MB/s eta 0:00:01
   ----------------------------- ---------- 8.9/12.0 MB 6.2 MB/s eta 0:00:01
   -------------------------------- ------- 9.7/12.0 MB 5.8 MB/s eta 0:00:01
   ---------------------------------- ----- 10.2/12.0 MB 5.4 MB/s eta 0:00:01
   ----------------------------------- ---- 10.7/12.0 MB 5.1 MB/s eta 0:00:01
   ------------------------------------- -- 11.3/12.0 MB 4.9 MB/s eta 0:00:01
   ---------------------------------------- 12.0/12.0 MB 4.7 MB/s eta 0:00:00
  

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

In [2]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")

In [3]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator
)

In [5]:
trainer.train()

Step,Training Loss
500,0.534800
1000,0.335800


TrainOutput(global_step=1377, training_loss=0.3744759874752491, metrics={'train_runtime': 2347.3232, 'train_samples_per_second': 4.688, 'train_steps_per_second': 0.587, 'total_flos': 405114969714960.0, 'train_loss': 0.3744759874752491, 'epoch': 3.0})

In [6]:
# Suppose you have some new examples
test_sentences = [
    {"sentence1": "The company is doing well.", "sentence2": "The company is performing excellently."},
    {"sentence1": "He likes football.", "sentence2": "He hates sports."}
]

# Tokenize them
test_encodings = tokenizer(
    [x["sentence1"] for x in test_sentences],
    [x["sentence2"] for x in test_sentences],
    truncation=True,
    padding=True,
    return_tensors="pt"
)

# Make predictions
import torch
model.eval()
with torch.no_grad():
    outputs = model(**test_encodings)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

print(predictions)  # tensor of predicted labels


tensor([0, 0])
